# Taller: de modelo de lenguaje a asistente mediante ajuste supervisado
**SI3014 · Redes Neuronales y Aprendizaje Profundo · Universidad EAFIT**

En este taller ajustarás un modelo de lenguaje base con ejemplos de instrucciones y respuestas en español. Usarás **LoRA**, `peft` y `SFTTrainer` de TRL; no tendrás que implementar el entrenamiento desde cero.

**Objetivos:** preparar ejemplos para SFT, configurar un adaptador LoRA, entrenar y comparar el modelo antes y después.

**Trabajo:** ejecuta las celdas en orden, completa los **7 TODO** y responde las preguntas en las celdas de texto. Conserva las salidas como evidencia. Algunas celdas se detendrán hasta completar el TODO anterior.

**Entorno:** activa una GPU (por ejemplo, T4 en Colab o Kaggle) y conexión a internet. Se descargan el modelo Qwen2.5-0.5B y los datos Dolly en español. Los tiempos dependen del entorno. La opción de CPU usa pocos ejemplos y puede ser lenta.

**Entrega:** este notebook con los TODO completados, las salidas visibles y las respuestas breves. La calidad del modelo no se evalúa solo por una respuesta individual.

## Paso 1. Preparar el entorno
Ejecuta la celda y verifica que el dispositivo sea `cuda`. No necesitas modificarla.

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import json, math, random, textwrap, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

try:
    import trl, peft
except ImportError:                                         # TRL no viene en Kaggle ni en Colab
    os.system('pip install -q "trl==1.16.0" peft')
    import trl, peft
import transformers, datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

random.seed(0); torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__, '·', dev)
if dev == 'cpu':
    print('Sin GPU: el notebook corre con muy pocos ejemplos. En Kaggle: Settings → Accelerator → GPU T4.')

## Paso 2. Cargar el modelo base
El modelo base predice el siguiente token. Aún no ha sido ajustado específicamente para responder instrucciones. Ejecuta la carga y observa la cantidad de parámetros.

In [ ]:
MODELO = 'Qwen/Qwen2.5-0.5B'          # el modelo BASE: solo preentrenado (token siguiente), sin SFT
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
cfg = modelo.config
N_TOTAL = modelo.num_parameters()
print(f'{MODELO}: {N_TOTAL / 1e6:.0f} M parámetros · {cfg.num_hidden_layers} bloques · '
      f'd = {cfg.hidden_size} · {cfg.num_attention_heads} cabezas · |V| = {len(tok):,} · fin: {tok.eos_token!r}')

### TODO 1 · Explorar el tokenizador
Escribe una frase en español y otra en inglés de significado similar. La celda cuenta sus tokens. Antes de ejecutarla, predice cuál necesitará más. Después explica lo observado; dos frases no permiten generalizar a todos los textos.

In [ ]:
# TODO 1: reemplaza los dos valores por frases de significado similar
frase_es = None
frase_en = None
assert isinstance(frase_es, str) and frase_es.strip(), "Escribe la frase en español"
assert isinstance(frase_en, str) and frase_en.strip(), "Escribe la frase en inglés"
for frase in [frase_es, frase_en]:
    ids = tok.encode(frase, add_special_tokens=False)
    print(f"{len(ids)} tokens: " + " | ".join(tok.decode([i]) for i in ids))

**Respuesta 1:** ¿Qué esperabas y qué observaste? Escribe aquí tu respuesta.

## Paso 3. Preparar el formato de entrenamiento
Cada ejemplo tendrá dos campos: `prompt`, que contiene la instrucción y el inicio de la respuesta, y `completion`, que contiene la respuesta esperada. TRL usa estos campos durante el entrenamiento.

In [ ]:
def prompt(pregunta):
    return f'Usuario: {pregunta.strip()}\nAsistente:'

print(prompt('¿Qué es un átomo?'))

### TODO 2 · Crear un ejemplo
Completa la función: usa `prompt` con el campo `instruction` y coloca el contenido de `response` en `completion`, precedido por un espacio. La prueba debe imprimir el diccionario con ambos campos.

In [ ]:
def a_ejemplo(e):
    # TODO 2: reemplaza los valores None
    return {'prompt': None, 'completion': None}

prueba = a_ejemplo({'instruction': '¿Qué es un átomo?', 'response': 'Una unidad básica de la materia.'})
assert isinstance(prueba['prompt'], str) and isinstance(prueba['completion'], str), 'Completa ambos campos'
print(prueba)

**Respuesta 2:** ¿Qué información recibe el modelo como entrada y qué parte debe aprender a generar?

## Paso 4. Observar el modelo antes del ajuste
Estas funciones generan respuestas y muestran si el modelo produjo el token de fin. Ejecuta la celda.

In [ ]:
@torch.no_grad()
def responder(m, pregunta, max_nuevos=120):
    """Greedy (diapositiva 46). Devuelve el texto y si el modelo produjo el token de fin."""
    x = tok(prompt(pregunta), return_tensors='pt').to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=tok.eos_token_id,
                     pad_token_id=tok.eos_token_id)
    nuevos = out[0, x['input_ids'].shape[1]:]
    return tok.decode(nuevos, skip_special_tokens=True).strip(), bool((nuevos == tok.eos_token_id).any())

def mostrar(q, r, paro, ancho=100):
    print('▶', q)
    print(textwrap.indent(textwrap.fill(r.replace('\n', ' ⏎ ')[:700], ancho), '   '))
    print('   se detuvo:', 'sí' if paro else 'NO, llegó al límite de tokens')
    print()

PREGUNTAS = ['¿Qué es un átomo?',
             'Dame tres ideas para ahorrar agua en casa.',
             '¿Cuál es la diferencia entre el clima y el tiempo atmosférico?',
             'Clasifica estos animales en mamíferos o aves: perro, águila, ballena, gorrión.',
             'Escribe un haiku sobre la lluvia.']

### TODO 3 · Probar el modelo base
Agrega una pregunta propia a la lista. Ejecuta la comparación inicial y observa si las respuestas son pertinentes y si terminan.

In [ ]:
# TODO 3: reemplaza None por una pregunta propia
pregunta_propia = None
assert isinstance(pregunta_propia, str) and pregunta_propia.strip(), 'Escribe una pregunta'
PREGUNTAS.append(pregunta_propia)
antes = {}
for q in PREGUNTAS:
    antes[q] = responder(modelo, q)
    mostrar(q, *antes[q])

**Respuesta 3:** Describe una respuesta y si el modelo se detuvo. ¿Qué limitación observas?

## Paso 5. Cargar y dividir los datos
Se usarán ejemplos Dolly traducidos automáticamente al español. Filtramos instrucciones que no requieren contexto adicional y respuestas de longitud moderada. La validación queda separada del entrenamiento.

In [ ]:
DOLLY = 'argilla/databricks-dolly-15k-curated-multilingual'
crudo = load_dataset(DOLLY, split='es')
print('Ejemplos originales:', len(crudo))
print('Categorías:', Counter(crudo['category']).most_common())

def util(e):
    return (not (e['context'] or '').strip() and 20 <= len(e['response']) <= 700
            and len(e['instruction']) <= 300)

filtrado = crudo.filter(util).shuffle(seed=0)
print('Ejemplos después del filtro:', len(filtrado))

### TODO 4 · Seleccionar y transformar los datos
Con GPU usa 2 000 ejemplos de entrenamiento y 200 de validación; sin GPU, 48 y 16. Completa los valores y ejecuta la celda. Los ejemplos se seleccionan sin solapamiento y luego se transforman con tu función.

In [ ]:
# TODO 4: completa las cantidades según el dispositivo
if dev == 'cuda':
    N_TRAIN, N_VAL = None, None
else:
    N_TRAIN, N_VAL = None, None
assert isinstance(N_TRAIN, int) and isinstance(N_VAL, int), 'Completa las cantidades'
assert N_TRAIN > 0 and N_VAL > 0 and N_TRAIN + N_VAL <= len(filtrado)
val_crudo = filtrado.select(range(N_VAL))
train_crudo = filtrado.select(range(N_VAL, N_VAL + N_TRAIN))
train = train_crudo.map(a_ejemplo, remove_columns=train_crudo.column_names)
val = val_crudo.map(a_ejemplo, remove_columns=val_crudo.column_names)
print('Entrenamiento:', len(train), '| Validación:', len(val))
print('Ejemplo:', train[0])

**Respuesta 4:** ¿Por qué se reservan ejemplos de validación y no se usan para actualizar los parámetros?

## Paso 6. Configurar LoRA y el entrenador
LoRA incorpora adaptadores entrenables de bajo rango, mientras mantiene congelados los pesos originales. TRL prepara los lotes y ejecuta el entrenamiento. Con `completion_only_loss=True`, la pérdida se calcula sobre la respuesta y no sobre el prompt.

### Configuración lista para ejecutar
La siguiente celda configura LoRA y el entrenador. **Ejecuta la celda sin modificarla**. LoRA permite ajustar un número reducido de parámetros y mantener congelados los pesos del modelo base. No necesitas conocer los detalles de su configuración para desarrollar el taller.

In [ ]:
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0,
                  task_type='CAUSAL_LM', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
args = SFTConfig(
    output_dir='sft_dolly', seed=0,
    completion_only_loss=True, max_length=384,
    num_train_epochs=1, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
    max_grad_norm=1.0, weight_decay=0.0, fp16=dev == 'cuda', bf16=False,
    eval_strategy='steps', eval_steps=50, logging_steps=10,
    save_strategy='no', report_to='none')
trainer = SFTTrainer(model=modelo, args=args, train_dataset=train, eval_dataset=val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()

**Observación:** La celda muestra cuántos parámetros se entrenarán. No es necesario modificar ni explicar la configuración de LoRA.

## Paso 7. Inspeccionar la máscara de pérdida
Las etiquetas con valor `-100` se excluyen de la pérdida. Ejecuta la celda y compara la secuencia completa con los tokens que cuentan.

In [ ]:
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
reales = lote['attention_mask'][0].bool() if 'attention_mask' in lote else torch.ones_like(ids, dtype=torch.bool)
print('entra al modelo:     ', repr(tok.decode(ids[reales])))
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))
print(f'{int((etiquetas != -100).sum())} de {int(reales.sum())} tokens tienen etiqueta; el resto vale -100 y no cuenta')

**Respuesta 6:** ¿Por qué es razonable calcular la pérdida sobre la respuesta en este taller? ¿El prompt deja de ser una entrada para el modelo?

## Paso 8. Evaluar y entrenar
Primero se mide la pérdida de validación del modelo base. Luego se entrena una época y se vuelve a medir. Una pérdida menor indica mejor predicción de las respuestas de validación según esta métrica, pero no garantiza respuestas correctas en todos los casos.

### TODO 5 · Medir antes del entrenamiento
Completa la llamada para evaluar el modelo y extraer `eval_loss`.

In [ ]:
# TODO 5: reemplaza None por la evaluación del entrenador
evaluacion_antes = None
assert isinstance(evaluacion_antes, dict) and 'eval_loss' in evaluacion_antes, 'Evalúa con trainer.evaluate()'
J_antes = evaluacion_antes['eval_loss']
print(f'NLL antes: {J_antes:.3f} · perplejidad: {math.exp(J_antes):.1f}')

### TODO 6 · Entrenar y evaluar
Completa las dos llamadas indicadas. El entrenamiento puede tardar varios minutos; no lo interrumpas si las métricas avanzan.

In [ ]:
t0 = time.time()
# TODO 6a: llama al método de entrenamiento de trainer
# Escribe tu código aquí

SEGUNDOS = time.time() - t0
# TODO 6b: evalúa el modelo entrenado y guarda el diccionario
evaluacion_despues = None
assert trainer.state.global_step > 0, 'Ejecuta el entrenamiento'
assert isinstance(evaluacion_despues, dict) and 'eval_loss' in evaluacion_despues, 'Evalúa después de entrenar'
J_despues = evaluacion_despues['eval_loss']
print(f'Tiempo: {SEGUNDOS/60:.1f} min | NLL: {J_antes:.3f} → {J_despues:.3f}')

Ejecuta la gráfica. La curva de validación tendrá pocos puntos si usas pocos ejemplos.

In [ ]:
log = trainer.state.log_history
tr = [(r['step'], r['loss']) for r in log if 'loss' in r]
ev = [(0, J_antes)] + [(r['step'], r['eval_loss']) for r in log if 'eval_loss' in r]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(*zip(*tr), label='entrenamiento')
ax.plot(*zip(*ev), 'o-', label='validación')
ax.axhline(J_antes, color='gray', ls='--', lw=1)
ax.set_xlabel('paso'); ax.set_ylabel('NLL de la respuesta (nats/token)'); ax.legend()
plt.tight_layout(); plt.show()

**Respuesta 7:** Compara las pérdidas antes y después. ¿Qué sugiere el resultado? ¿Qué no puedes concluir solo a partir de esta métrica?

## Paso 9. Comparar respuestas
Se utilizan las mismas preguntas y parámetros de generación del paso 4. Observa pertinencia, claridad y si el modelo se detiene.

### TODO 7 · Generar con el modelo ajustado
Completa la llamada a `responder` dentro del ciclo con el modelo ajustado `m` y la pregunta `q`.

In [ ]:
m = trainer.model
m.eval()
despues = {}
for q in PREGUNTAS:
    # TODO 7: genera y guarda la respuesta con el modelo ajustado
    despues[q] = None
    assert isinstance(despues[q], tuple), 'Usa responder(m, q)'
    print('ANTES  ', end=''); mostrar(q, *antes[q])
    print('DESPUÉS', end=''); mostrar(q, *despues[q])
    print('-' * 80)

**Respuesta 8:** Compara al menos dos preguntas, incluida la propia. Indica un cambio observado y una limitación que permanezca. ¿Cambió la capacidad de detenerse?

## Cierre
1. Explica con tus palabras la diferencia entre un modelo base y uno ajustado mediante SFT.
2. ¿Por qué es útil comparar el modelo antes y después del ajuste?
3. Si quisieras evaluar mejor el asistente, ¿qué ejemplos o criterios adicionales usarías?

**Respuestas:** escribe aquí tus conclusiones.

### Referencias
- [Datos Dolly multilingües](https://huggingface.co/datasets/argilla/databricks-dolly-15k-curated-multilingual).
- [Documentación de TRL](https://huggingface.co/docs/trl) y [PEFT](https://huggingface.co/docs/peft).
- Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*; Ouyang et al. (2022), *Training language models to follow instructions with human feedback*.